# checagem do dataset - spacenet 2 aoi_3 paris

esse notebook explora o dataset de verdade: estrutura de pastas, formato dos
arquivos, por que os .tif abrem preto, e como a gente transforma o geojson
em mascara pro treino.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.features import rasterize as rasterio_rasterize
from PIL import Image

# troque pro caminho do dataset na sua maquina
DATA_DIR = '/mnt/c/Users/Cliente/Desktop/claude/geojson/spacenet'

## 1. estrutura de pastas

o dataset vem organizado assim. a gente usa so o `RGB-PanSharpen` (3 bandas, colorido).

In [ ]:
for split in ['train', 'test']:
    split_dir = os.path.join(DATA_DIR, split)
    if not os.path.isdir(split_dir):
        continue
    print(f'{split}/')
    for sub in sorted(os.listdir(split_dir)):
        sub_path = os.path.join(split_dir, sub)
        if os.path.isdir(sub_path):
            # conta arquivos dentro
            if sub == 'geojson':
                buildings_dir = os.path.join(sub_path, 'buildings')
                n = len(os.listdir(buildings_dir)) if os.path.isdir(buildings_dir) else 0
                print(f'  {sub}/buildings/ ({n} arquivos)')
            else:
                n = len([f for f in os.listdir(sub_path) if os.path.isfile(os.path.join(sub_path, f))])
                print(f'  {sub}/ ({n} arquivos)')

## 2. por que o .tif abre preto?

as imagens sao **16-bit** (valores de 0 a 65535), mas os pixels reais ficam
espremidos la embaixo (tipo 100 a 500). quando o visualizador mapeia
0-65535 pra tela, esses valores baixos viram quase zero = preto.

a solucao: **normalizacao percentil 2-98** - pega a faixa real dos valores
e estica pra 0-255.

In [ ]:
# abre um tile de exemplo
tile_path = os.path.join(DATA_DIR, 'train', 'RGB-PanSharpen', 'RGB-PanSharpen_AOI_3_Paris_img485.tif')

with rasterio.open(tile_path) as src:
    bands = [src.read(i + 1) for i in range(3)]
    img_raw = np.stack(bands, axis=-1)
    crs = src.crs
    transform = src.transform

print(f'dimensao: {img_raw.shape[1]}x{img_raw.shape[0]} pixels')
print(f'bandas: {img_raw.shape[2]}')
print(f'dtype: {img_raw.dtype}')
print(f'valor min: {img_raw.min()}')
print(f'valor max: {img_raw.max()} (de 65535 possiveis)')
print(f'percentil 2: {np.percentile(img_raw, 2):.0f}')
print(f'percentil 98: {np.percentile(img_raw, 98):.0f}')
print(f'crs: {crs}')

In [ ]:
# comparacao: crua vs normalizada
img_float = img_raw.astype(np.float32)

# crua: dividir por 256 (o que o visualizador faz mais ou menos)
img_crua = np.clip(img_float / 256.0, 0, 255).astype(np.uint8)

# normalizada: percentil 2-98
p2, p98 = np.percentile(img_float, 2), np.percentile(img_float, 98)
img_norm = np.clip((img_float - p2) / (p98 - p2), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_crua)
axes[0].set_title('crua (como o visualizador ve - preto)')
axes[0].axis('off')
axes[1].imshow(img_norm)
axes[1].set_title('normalizada (percentil 2-98)')
axes[1].axis('off')
plt.suptitle('por que o .tif abre preto', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. de .tif pra mascara: o caminho completo

o dataset tem dois arquivos por tile:
- `.tif` = a foto de satelite (a entrada do modelo, o X)
- `.geojson` = os contornos dos predios em lat/lon (o gabarito)

mas a rede neural precisa do gabarito como **imagem** (mascara 0/1),
nao como poligonos. entao a gente **rasteriza**: pinta os poligonos
numa grade de pixels, branco = predio, preto = fundo.

In [ ]:
# carrega o geojson correspondente
geojson_path = os.path.join(DATA_DIR, 'train', 'geojson', 'buildings', 'buildings_AOI_3_Paris_img485.geojson')
gdf = gpd.read_file(geojson_path)

print(f'arquivo: {os.path.basename(geojson_path)}')
print(f'predios: {len(gdf)}')
print(f'crs: {gdf.crs}')
print(f'tipo geometria: {gdf.geom_type.unique()}')
print()
print('primeiros 3 predios:')
gdf.head(3)

In [ ]:
# rasteriza: poligonos -> mascara binaria
valid_geoms = [
    geom for geom in gdf.geometry
    if geom is not None and geom.is_valid and not geom.is_empty
]
shapes = [(geom, 1) for geom in valid_geoms]

mask = rasterio_rasterize(
    shapes,
    out_shape=(img_raw.shape[0], img_raw.shape[1]),
    transform=transform,
    fill=0,
    dtype=np.uint8,
)

print(f'mascara shape: {mask.shape}')
print(f'pixels de predio: {mask.sum()} de {mask.size} ({100*mask.sum()/mask.size:.1f}%)')

In [ ]:
# visualiza o caminho completo: imagem -> mascara -> overlay
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_norm)
axes[0].set_title('imagem (.tif normalizado)')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title(f'mascara ({len(valid_geoms)} predios rasterizados)')
axes[1].axis('off')

overlay = img_norm.copy()
overlay[mask == 1] = overlay[mask == 1] * 0.5 + np.array([1, 0.3, 0.3]) * 0.5
axes[2].imshow(overlay)
axes[2].set_title('overlay (predios em vermelho)')
axes[2].axis('off')

plt.suptitle('.tif -> geojson -> mascara -> overlay', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. distribuicao de predios por tile

muitos tiles nao tem nenhum predio (geojson com `features: []` vazio).
isso e normal e valido - o modelo precisa aprender a dizer "nao tem".

In [ ]:
import re

geojson_dir = os.path.join(DATA_DIR, 'train', 'geojson', 'buildings')
counts = []

for f in sorted(os.listdir(geojson_dir)):
    if not f.endswith('.geojson'):
        continue
    try:
        g = gpd.read_file(os.path.join(geojson_dir, f))
        n = len([x for x in g.geometry if x is not None and x.is_valid and not x.is_empty])
    except:
        n = 0
    counts.append(n)

counts = np.array(counts)
print(f'total de tiles: {len(counts)}')
print(f'tiles com 0 predios: {(counts == 0).sum()} ({100*(counts == 0).mean():.1f}%)')
print(f'tiles com predios: {(counts > 0).sum()}')
print(f'media de predios/tile: {counts.mean():.1f}')
print(f'max predios num tile: {counts.max()}')
print(f'mediana: {np.median(counts):.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(counts, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('predios por tile')
axes[0].set_ylabel('quantidade de tiles')
axes[0].set_title('distribuicao de predios por tile')
axes[0].axvline(counts.mean(), color='red', linestyle='--', label=f'media: {counts.mean():.1f}')
axes[0].legend()

axes[1].hist(counts[counts > 0], bins=40, color='coral', edgecolor='white')
axes[1].set_xlabel('predios por tile')
axes[1].set_ylabel('quantidade de tiles')
axes[1].set_title('so tiles com predios (sem os vazios)')

plt.tight_layout()
plt.show()

## 5. tile vazio vs tile cheio

comparacao visual de um tile sem nenhum predio vs um com muitos.

In [ ]:
# acha um tile vazio e um cheio
geojson_files = sorted(os.listdir(geojson_dir))
tile_vazio = None
tile_cheio = None
max_predios = 0

for f in geojson_files:
    if not f.endswith('.geojson'):
        continue
    num = re.search(r'img(\d+)', f).group(1)
    try:
        g = gpd.read_file(os.path.join(geojson_dir, f))
        n = len([x for x in g.geometry if x is not None and x.is_valid and not x.is_empty])
    except:
        n = 0
    if n == 0 and tile_vazio is None:
        tile_vazio = num
    if n > max_predios:
        max_predios = n
        tile_cheio = num

def load_and_normalize(tile_num):
    path = os.path.join(DATA_DIR, 'train', 'RGB-PanSharpen', f'RGB-PanSharpen_AOI_3_Paris_img{tile_num}.tif')
    with rasterio.open(path) as src:
        bands = [src.read(i+1) for i in range(3)]
        img = np.stack(bands, axis=-1).astype(np.float32)
    p2, p98 = np.percentile(img, 2), np.percentile(img, 98)
    return np.clip((img - p2) / (p98 - p2), 0, 1) if p98 > p2 else np.zeros_like(img)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(load_and_normalize(tile_vazio))
axes[0].set_title(f'tile vazio (img{tile_vazio}, 0 predios)')
axes[0].axis('off')

axes[1].imshow(load_and_normalize(tile_cheio))
axes[1].set_title(f'tile mais cheio (img{tile_cheio}, {max_predios} predios)')
axes[1].axis('off')

plt.suptitle('tile vazio vs tile cheio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. as 4 modalidades do spacenet

cada tile tem 4 versoes. a gente usa so o RGB-PanSharpen (3 bandas, colorido).

In [ ]:
tile_num = '485'
modalities = {
    'RGB-PanSharpen': f'RGB-PanSharpen_AOI_3_Paris_img{tile_num}.tif',
    'PAN': f'PAN_AOI_3_Paris_img{tile_num}.tif',
    'MUL': f'MUL_AOI_3_Paris_img{tile_num}.tif',
    'MUL-PanSharpen': f'MUL-PanSharpen_AOI_3_Paris_img{tile_num}.tif',
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, (name, filename) in enumerate(modalities.items()):
    path = os.path.join(DATA_DIR, 'train', name, filename)
    if not os.path.exists(path):
        axes[idx].set_title(f'{name} (nao encontrado)')
        axes[idx].axis('off')
        continue
    with rasterio.open(path) as src:
        n_bands = src.count
        h, w = src.height, src.width
        # pega ate 3 bandas pra visualizar
        bands = [src.read(i+1) for i in range(min(3, n_bands))]
        if len(bands) == 1:
            img = bands[0].astype(np.float32)
        else:
            img = np.stack(bands, axis=-1).astype(np.float32)
    
    p2, p98 = np.percentile(img, 2), np.percentile(img, 98)
    if p98 > p2:
        img = np.clip((img - p2) / (p98 - p2), 0, 1)
    
    if img.ndim == 2:
        axes[idx].imshow(img, cmap='gray')
    else:
        axes[idx].imshow(img)
    axes[idx].set_title(f'{name}\n{n_bands} bandas, {h}x{w}')
    axes[idx].axis('off')

plt.suptitle(f'as 4 modalidades do tile img{tile_num}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()